# Check ensemble padding

Notebook minimo para debuggear el flujo de generacion entre `src/ensemble.py` y `DiffusionModel._prepare_backbone_inputs` usando `lengths`.


In [1]:
from pathlib import Path
import os
import sys
import types

import torch as tr

def repo_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'src').exists() and (p / 'data').exists():
            return p.resolve()
    raise RuntimeError('Run this notebook from inside the repo.')

ROOT = repo_root()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src.config import load_config
from src.data import build_dataloader
from src.ensemble import _sample_batch
from src.io import load_model_checkpoint

tr.set_printoptions(precision=4, sci_mode=False)
DEVICE = tr.device('cuda' if tr.cuda.is_available() else 'cpu')
print('ROOT =', ROOT)
print('DEVICE =', DEVICE)


ROOT = /home/gkulemeyer/Documents/Repos/RNADiffusion
DEVICE = cuda


In [3]:
checkpoint_candidates = sorted(
    ROOT.glob('logs/**/checkpoints/best.ckpt'),
    key=lambda p: p.stat().st_mtime,
)
assert checkpoint_candidates, 'No encontre checkpoints best.ckpt en logs/**/checkpoints/'

CHECKPOINT_PATH = checkpoint_candidates[-1]
EXPERIMENT_DIR = CHECKPOINT_PATH.parent.parent
CONFIG_PATH = EXPERIMENT_DIR / 'config.yaml'

print('CHECKPOINT_PATH =', CHECKPOINT_PATH)
print('EXPERIMENT_DIR =', EXPERIMENT_DIR)
print('CONFIG_PATH =', CONFIG_PATH)

config = load_config(CONFIG_PATH)
config['training']['batch_size'] = max(2, int(config['training'].get('batch_size', 4)))
config['training']['num_workers'] = 0

model = load_model_checkpoint(config, CHECKPOINT_PATH, eval_mode=True)
print('model class =', model.__class__.__name__)
print('model device =', next(model.parameters()).device)


CHECKPOINT_PATH = /home/gkulemeyer/Documents/Repos/RNADiffusion/logs/ArchiveII_test_newexp/sim70/t25/m64/20260330-1907_model_64_Fold0/checkpoints/best.ckpt
EXPERIMENT_DIR = /home/gkulemeyer/Documents/Repos/RNADiffusion/logs/ArchiveII_test_newexp/sim70/t25/m64/20260330-1907_model_64_Fold0
CONFIG_PATH = /home/gkulemeyer/Documents/Repos/RNADiffusion/logs/ArchiveII_test_newexp/sim70/t25/m64/20260330-1907_model_64_Fold0/config.yaml
model class = DiffusionModel
model device = cuda:0


In [4]:
loader = build_dataloader(config, partition='test', shuffle=False)
batch = next(iter(loader))

print('batch keys =', sorted(batch.keys()))
print('batch lengths =', batch['length'])
print('conditioning shape =', tuple(batch['conditioning'].shape))
print('contact_oh shape =', tuple(batch['contact_oh'].shape))

assert len(batch['length']) >= 2, 'Necesito al menos 2 secuencias en el batch para probar repeat_interleave.'

conditioning = batch['conditioning'][:2].to(DEVICE)
targets = batch['contact_oh'][:2].to(DEVICE)
lengths = tr.tensor(batch['length'][:2], dtype=tr.long, device=DEVICE)

print('selected lengths =', lengths.tolist())
print('selected conditioning shape =', tuple(conditioning.shape))
print('selected targets shape =', tuple(targets.shape))


batch keys = ['conditioning', 'contact', 'contact_oh', 'embedding', 'id', 'length', 'mask', 'sequence']
batch lengths = [107, 75, 74, 54]
conditioning shape = (4, 16, 108, 108)
contact_oh shape = (4, 2, 108, 108)
selected lengths = [107, 75]
selected conditioning shape = (2, 16, 108, 108)
selected targets shape = (2, 2, 108, 108)


In [5]:
orig_prepare = model._prepare_backbone_inputs

def debug_prepare(self, xt_input, condition, lengths=None):
    print('\n[_prepare_backbone_inputs]')
    print('  xt_input shape =', tuple(xt_input.shape))
    print('  condition shape =', tuple(condition.shape))
    if lengths is None:
        print('  lengths = None')
    else:
        lengths_t = tr.as_tensor(lengths, device=condition.device, dtype=tr.long)
        print('  lengths shape =', tuple(lengths_t.shape), 'values =', lengths_t.tolist())
        mask = self._lengths_to_mask(lengths_t, condition.shape[-1], device=condition.device)
        print('  derived mask shape =', tuple(mask.shape))
        print('  batch match =', mask.shape[0] == condition.shape[0])
    xt_backbone, cond_backbone = orig_prepare(xt_input, condition, lengths=lengths)
    print('  xt_backbone shape =', tuple(xt_backbone.shape))
    print('  cond_backbone shape =', tuple(cond_backbone.shape))
    if lengths is not None:
        l0 = int(tr.as_tensor(lengths, device=condition.device, dtype=tr.long)[0].item())
        if l0 < xt_backbone.shape[-1]:
            print('  xt padded pixel sample  =', xt_backbone[0, :, l0, l0].detach().cpu())
            print('  cond padded pixel sample =', cond_backbone[0, :4, l0, l0].detach().cpu())
    return xt_backbone, cond_backbone

model._prepare_backbone_inputs = types.MethodType(debug_prepare, model)
print('debug wrapper installed')


debug wrapper installed


In [8]:
current_chunk = 3
expanded_conditioning = conditioning.repeat_interleave(current_chunk, dim=0)
expanded_lengths = lengths.repeat_interleave(current_chunk)
dummy_xt = tr.zeros(
    (expanded_conditioning.shape[0], expanded_conditioning.shape[-2], expanded_conditioning.shape[-1]),
    device=DEVICE,
    dtype=tr.long,
)

print('manual ensemble-style expansion')
print('  expanded_conditioning shape =', tuple(expanded_conditioning.shape))
print('  expanded_lengths shape =', tuple(expanded_lengths.shape), 'values =', expanded_lengths.tolist())
print('  batch sizes match =', expanded_conditioning.shape[0] == expanded_lengths.shape[0])

xt_backbone, cond_backbone = model._prepare_backbone_inputs(
    dummy_xt,
    expanded_conditioning,
    lengths=expanded_lengths,
)

print('\nmanual call completed')
print('  xt_backbone dtype =', xt_backbone.dtype)
print('  cond_backbone dtype =', cond_backbone.dtype)


manual ensemble-style expansion
  expanded_conditioning shape = (6, 16, 108, 108)
  expanded_lengths shape = (6,) values = [107, 107, 107, 75, 75, 75]
  batch sizes match = True

[_prepare_backbone_inputs]
  xt_input shape = (6, 108, 108)
  condition shape = (6, 16, 108, 108)
  lengths shape = (6,) values = [107, 107, 107, 75, 75, 75]
  derived mask shape = (6, 1, 108, 108)
  batch match = True
  xt_backbone shape = (6, 2, 108, 108)
  cond_backbone shape = (6, 16, 108, 108)
  xt padded pixel sample  = tensor([1., 0.])
  cond padded pixel sample = tensor([0., 0., 0., 0.])

manual call completed
  xt_backbone dtype = torch.float32
  cond_backbone dtype = torch.float32


In [9]:
NUM_SAMPLES = 3
CHUNK_SIZE = 2
BASE_SEED = 123

samples = _sample_batch(
    model=model,
    conditioning=conditioning,
    lengths=lengths,
    num_samples=NUM_SAMPLES,
    base_seed=BASE_SEED,
    chunk_size=CHUNK_SIZE,
)

print('\n_sample_batch output')
print('  samples shape =', tuple(samples.shape))
print('  samples dtype =', samples.dtype)
print('  expected shape =', (conditioning.shape[0], NUM_SAMPLES, conditioning.shape[-2], conditioning.shape[-1]))
print('  shape ok =', tuple(samples.shape) == (conditioning.shape[0], NUM_SAMPLES, conditioning.shape[-2], conditioning.shape[-1]))

for i, length in enumerate(lengths.tolist()):
    padded_value = None
    if length < samples.shape[-1]:
        padded_value = int(samples[i, 0, length, length].item())
    print(f'  seq {i}: length={length}, padded sample value at [length, length] = {padded_value}')



[_prepare_backbone_inputs]
  xt_input shape = (4, 108, 108)
  condition shape = (4, 16, 108, 108)
  lengths shape = (4,) values = [107, 107, 75, 75]
  derived mask shape = (4, 1, 108, 108)
  batch match = True
  xt_backbone shape = (4, 2, 108, 108)
  cond_backbone shape = (4, 16, 108, 108)
  xt padded pixel sample  = tensor([1., 0.])
  cond padded pixel sample = tensor([0., 0., 0., 0.])

[_prepare_backbone_inputs]
  xt_input shape = (4, 108, 108)
  condition shape = (4, 16, 108, 108)
  lengths shape = (4,) values = [107, 107, 75, 75]
  derived mask shape = (4, 1, 108, 108)
  batch match = True
  xt_backbone shape = (4, 2, 108, 108)
  cond_backbone shape = (4, 16, 108, 108)
  xt padded pixel sample  = tensor([1., 0.])
  cond padded pixel sample = tensor([0., 0., 0., 0.])

[_prepare_backbone_inputs]
  xt_input shape = (4, 108, 108)
  condition shape = (4, 16, 108, 108)
  lengths shape = (4,) values = [107, 107, 75, 75]
  derived mask shape = (4, 1, 108, 108)
  batch match = True
  xt_ba